In [1]:
from google.colab import drive
drive.mount('/content/drive')
db_dir = "/content/drive/MyDrive/Partial Credit DB"


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.1 MB/s eta 0:00:00


In [ ]:
#code similarity library
!pip install deepcsim
!pip install python-Levenshtein
import Levenshtein as lev
from deepcsim import compare_source

def cleanCode(code):
    if not code: return ""
    lines = []
    for line in code.strip().splitlines():
        if 'import' in line: continue
        if '@testFunction' in line or 'def test' in line: break
        lines.append(line)
    return '\n'.join(lines)

def compareCode(orig_raw, corr_raw):
    if orig_raw == "NO UNCORRECTED CODE":
        return (0, "No original submission to compare.")
    c1, c2 = cleanCode(orig_raw), cleanCode(corr_raw)
    try:
        report = compare_source(c1, c2)
        return (report['results'][0]['similarity'], '')
    except:
        ratio = lev.ratio(c1, c2)
        return (round(ratio * 50), 'Possible syntax error (Levenshtein fallback)')

def cleanCode(code):
    if not code: return ""
    lines = []
    for line in code.strip().splitlines():
        if 'import' in line: continue
        if '@testFunction' in line or 'def test' in line: break
        lines.append(line)
    return '\n'.join(lines)

def compareCode(orig_raw, corr_raw):
    if orig_raw == "NO UNCORRECTED CODE":
        return (0, "No original submission to compare.")
    c1, c2 = cleanCode(orig_raw), cleanCode(corr_raw)
    try:
        report = compare_source(c1, c2)
        return (report['results'][0]['similarity'], '')
    except:
        ratio = lev.ratio(c1, c2)
        return (round(ratio * 50), 'Possible syntax error (Levenshtein fallback)')

In [ ]:
import sqlite3
import os
import re
import glob
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build

# --- 1. SETUP & AUTHENTICATION ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

DB_PATH = "/content/drive/MyDrive/Partial Credit DB/ws.db"
BASE_SUBMISSIONS_DIR = "/content/drive/MyDrive/Partial Credit DB/original-submissions"
RESPONSES_SHEET_NAME = "[S26] Writing Session Partial Credit Form (Responses)"
EXPORT_SHEET_NAME = "ws-partial-credit"

# --- 2. HELPER FUNCTIONS ---

def get_folder_id(folder_name):
    query = f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    items = results.get('files', [])
    return items[0]['id'] if items else None

def get_or_create_spreadsheet(title, folder_id):
    try:
        return gc.open(title)
    except gspread.SpreadsheetNotFound:
        print(f"Creating new spreadsheet: '{title}'")
        meta = {'name': title, 'mimeType': 'application/vnd.google-apps.spreadsheet', 'parents': [folder_id]}
        file = drive_service.files().create(body=meta, fields='id').execute()
        return gc.open_by_key(file.get('id'))



# --- 3. CORE OPERATIONS ---

def init_db():
    confirm = input("Are you sure you want to RESET the database? (y/n): ").lower()
    if confirm != 'y': return
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("DROP TABLE IF EXISTS submissions")
    cur.execute("""
        CREATE TABLE submissions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            wsNum INTEGER, problemName TEXT, version TEXT, problemPoints INTEGER,
            andrewID TEXT, originalCode TEXT, correctedCode TEXT,
            correctedIsCorrect INTEGER DEFAULT 0,
            partialCreditPct REAL DEFAULT 0.0 CHECK(partialCreditPct >= 0 AND partialCreditPct <= 100),
            partialCreditPoints REAL DEFAULT 0.0, note TEXT DEFAULT "",
            UNIQUE(wsNum, problemName, version, andrewID)
        )""")
    con.commit()
    con.close()
    print("Database Reset Successful.")

def import_original(target_ws):
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    pattern = f"{BASE_SUBMISSIONS_DIR}/original-ws*/Version */*/*.py"
    count = 0
    for path in glob.glob(pattern):
        if '-corrected' in path: continue
        parts = path.replace('\\', '/').split('/')
        p_name, v = parts[-2], parts[-3].replace('Version ', '').strip()
        ws = int(parts[-4].replace('original-ws', '').strip())
        a_id = parts[-1].split('-')[0]

        if target_ws is None or ws == target_ws:
            with open(path, "r", encoding="utf-8") as f:
                code = f.read()
            cur.execute("""INSERT OR REPLACE INTO submissions (wsNum, problemName, version, problemPoints, andrewID, originalCode)
                           VALUES (?, ?, ?, 10, ?, ?)""", (ws, p_name, v, a_id, code))
            count += 1
    con.commit()
    con.close()
    print(f"Imported {count} original submissions.")

def import_corrected(target_ws):
    con = sqlite3.connect(DB_PATH); cur = con.cursor()
    ss = gc.open(RESPONSES_SHEET_NAME)
    tabs = [ss.worksheet(f"ws{target_ws}")] if target_ws else [s for s in ss.worksheets() if s.title.startswith('ws')]
    total = 0
    for ws_sheet in tabs:
        ws_num = int(ws_sheet.title.replace('ws', ''))
        data = ws_sheet.get_all_values()
        headers = data[0]
        idx_a = next(i for i, h in enumerate(headers) if h.strip().lower() == 'andrewid')
        prob_cols = []
        for i in range(idx_a + 1, len(headers)):
            m = re.match(r"^(.*)([A-Z])$", headers[i].strip())
            if m: prob_cols.append((i, m.group(1), m.group(2)))

        for row in data[1:]:
            a_id = row[idx_a].strip()
            if not a_id: continue
            for c_idx, p_name, v in prob_cols:
                code = row[c_idx].strip() if c_idx < len(row) else ""
                if code:
                    cur.execute("UPDATE submissions SET correctedCode=?, correctedIsCorrect=1 WHERE wsNum=? AND problemName=? AND version=? AND andrewID=?", (code, ws_num, p_name, v, a_id))
                    if cur.rowcount == 0:
                        print(f"\n[!] Missing original for {a_id} - {p_name}{v}")
                        if input("Create new record with 'NO UNCORRECTED CODE'? (y/n): ").lower() == 'y':
                            cur.execute("INSERT INTO submissions (wsNum, problemName, version, problemPoints, andrewID, originalCode, correctedCode, correctedIsCorrect) VALUES (?,?,?,10,?,?,?,1)", (ws_num, p_name, v, a_id, "NO UNCORRECTED CODE", code))
                    total += 1
    con.commit(); con.close()
    print(f"Updated {total} corrections.")

def run_scoring(target_ws):
    con = sqlite3.connect(DB_PATH); cur = con.cursor()
    q = "SELECT id, originalCode, correctedCode, problemPoints, note FROM submissions WHERE correctedCode IS NOT NULL"
    if target_ws: q += f" AND wsNum = {target_ws}"
    cur.execute(q)
    rows = cur.fetchall()
    updates = []
    for r_id, orig, corr, pts, old_note in rows:
        score, note = compareCode(orig, corr)
        pct = (score / 50.0) * 100.0 if "Levenshtein" in note else float(score)
        calc_pts = round((pct / 100.0) * pts, 2)
        updates.append((pct, calc_pts, note, r_id))
    cur.executemany("UPDATE submissions SET partialCreditPct=?, partialCreditPoints=?, note=? WHERE id=?", updates)
    con.commit(); con.close()
    print(f"Scored {len(updates)} records.")

def export_to_sheets(target_ws):
    parent_folder = os.path.basename(os.getcwd())
    if parent_folder == "content": parent_folder = "Partial Credit DB"
    f_id = get_folder_id(parent_folder)
    sh = get_or_create_spreadsheet(EXPORT_SHEET_NAME, f_id)
    con = sqlite3.connect(DB_PATH)

    ws_list = [target_ws] if target_ws else [r[0] for r in con.execute("SELECT DISTINCT wsNum FROM submissions").fetchall()]
    for ws in ws_list:
        # Require correctedCode IS NOT NULL to ensure we only pull students who submitted for partial credit
        df = pd.read_sql_query(f"SELECT andrewID, problemName, version, partialCreditPct, partialCreditPoints, note FROM submissions WHERE wsNum={ws} AND correctedCode IS NOT NULL AND originalCode != 'NO UNCORRECTED CODE'", con)
        if df.empty: continue
        title = f"ws{ws}-partial-credit"
        try:
            wks = sh.worksheet(title)
            wks.clear() # Clear existing data
        except gspread.WorksheetNotFound:
            wks = sh.add_worksheet(title, 100, 10)
        wks.update([df.columns.values.tolist()] + df.values.tolist())
        print(f"Exported {title}")
    con.close()

def manage_student_submissions():
    while True:
        print("\n--- MANAGE STUDENT SUBMISSIONS ---")
        andrew_id = input("Enter the student's andrewID (or 'back' to return to main menu): ").strip()
        if andrew_id.lower() == 'back':
            return

        ws_num_str = input(f"Enter the WS number to search for '{andrew_id}' (e.g., '1', or 'back' to search a different student): ").strip()
        if ws_num_str.lower() == 'back':
            continue

        try:
            ws_num = int(ws_num_str)
        except ValueError:
            print("Invalid WS number.")
            continue

        con = sqlite3.connect(DB_PATH)
        cur = con.cursor()

        while True:
            cur.execute("SELECT id, problemName, version, originalCode, correctedCode FROM submissions WHERE andrewID = ? AND wsNum = ?", (andrew_id, ws_num))
            rows = cur.fetchall()

            if not rows:
                print(f"\nNo submissions found for '{andrew_id}' in WS{ws_num}.")
                break

            print(f"\nSubmissions for {andrew_id} in WS{ws_num}:")
            for i, row in enumerate(rows):
                print(f"{i + 1}. Problem: {row[1]} (Version {row[2]})")
            print("0. Go Back (to student selection)")

            choice = input("\nSelect a problem to view/delete (0 to go back): ").strip()
            if choice == '0':
                break

            try:
                idx = int(choice) - 1
                if idx < 0 or idx >= len(rows):
                    print("Invalid selection.")
                    continue

                selected_row = rows[idx]
                r_id, p_name, v, orig, corr = selected_row

                print(f"\n{'='*40}")
                print(f"--- {p_name} (Version {v}) ---")
                print(f"{'='*40}")
                print("\n>>> ORIGINAL CODE:")
                print(orig if orig else "None")
                print("\n>>> CORRECTED CODE:")
                print(corr if corr else "None")
                print("-" * 40)

                action = input("\nType 'delete' to remove this submission, or press Enter to return to the problem list: ").strip().lower()
                if action == 'delete':
                    confirm = input(f"Are you sure you want to permanently delete {p_name} for {andrew_id}? (y/n): ").lower()
                    if confirm == 'y':
                        cur.execute("DELETE FROM submissions WHERE id = ?", (r_id,))
                        con.commit()
                        print(f"\n[SUCCESS] Deleted '{p_name}' submission for {andrew_id}.")
            except ValueError:
                print("Invalid input.")

        con.close()

# --- 4. MAIN CLI LOOP ---

def main_menu():
    while True:
        print("\n" + "="*40)
        print("   PARTIAL CREDIT MASTER MENU")
        print("="*40)
        print("1. Initialize/Reset Database")
        print("2. Import Original Submissions (Drive)")
        print("3. Import Corrected Submissions (Sheets)")
        print("4. Generate Similarity Scores")
        print("5. Export Results to Spreadsheet")
        print("6. Manage Specific Student Submissions")
        print("7. Exit")

        choice = input("\nSelect an operation (1-7): ").strip()

        if choice == '7':
            print("Exiting. Have a great day!")
            break
        if choice == '1':
            init_db()
            continue
        if choice == '6':
            manage_student_submissions()
            continue

        scope = input("Enter WS number or 'all' (or 'back' to cancel): ").strip().lower()
        if scope == 'back': continue
        ws_num = int(scope) if scope != 'all' else None

        if choice == '2': import_original(ws_num)
        elif choice == '3': import_corrected(ws_num)
        elif choice == '4': run_scoring(ws_num)
        elif choice == '5': export_to_sheets(ws_num)
        else: print("Invalid selection. Please choose a valid option.")

if __name__ == "__main__":
    main_menu()


   PARTIAL CREDIT MASTER MENU
1. Initialize/Reset Database
2. Import Original Submissions (Drive)
3. Import Corrected Submissions (Sheets)
4. Generate Similarity Scores
5. Export Results to Spreadsheet
6. Manage Specific Student Submissions
7. Exit

Select an operation (1-7): 6

--- MANAGE STUDENT SUBMISSIONS ---
Enter the student's andrewID (or 'back' to cancel): eochis
Enter the WS number to search for 'eochis' (e.g., '1', or 'back'): 1

No submissions found for 'eochis' in WS1.

   PARTIAL CREDIT MASTER MENU
1. Initialize/Reset Database
2. Import Original Submissions (Drive)
3. Import Corrected Submissions (Sheets)
4. Generate Similarity Scores
5. Export Results to Spreadsheet
6. Manage Specific Student Submissions
7. Exit

Select an operation (1-7): 5
Enter WS number or 'all' (or 'back' to cancel): 1


Exported ws1-partial-credit

   PARTIAL CREDIT MASTER MENU
1. Initialize/Reset Database
2. Import Original Submissions (Drive)
3. Import Corrected Submissions (Sheets)
4. Generate Similarity Scores
5. Export Results to Spreadsheet
6. Manage Specific Student Submissions
7. Exit

Select an operation (1-7): 7
Exiting. Have a great day!
